In [1]:
import warnings
warnings.filterwarnings("ignore", message="The default value of `allowed_objects`")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

/mnt/c/Users/wstre/projects/lca-lc-foundations/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [13]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients or an image that contains ingredients they have at hand.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [14]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    temperature=0.8,
)

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [15]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken,rice, an onion, and some tortillas. What can I make?")]},
    config
)

print(response['messages'][-1].content)

Here are some recipe ideas you can make with your ingredients:

1. **Crispy Chicken Quesadillas**  
   - Use tortillas, shredded chicken, and sautéed onions (and optional peppers/corn if available). Top with cheese and crisp in a pan.  
   - *Need*: Tortillas, cheese (optional).  

2. **Chicken Fajita Tacos**  
   - Sauté chicken, onions, and peppers (if available), then serve in tortillas with rice on the side.  
   - *Need*: Tortillas, optional peppers/cheese.  

3. **Chicken & Rice Burritos**  
   - Mix shredded chicken, rice, sautéed onions, and seasonings (e.g., taco seasoning), then wrap in tortillas.  
   - *Need*: Taco seasoning, cheese (optional).  

4. **Chicken Tortilla Rice Bake** (Casserole-style)  
   - Combine chicken, rice, tomatoes, garlic, and corn tortillas, then bake with cheese.  
   - *Need*: Tomatoes, garlic, cheese (substitute onion for garlic if needed).  

Would you like detailed instructions for any of these? Let me know!


In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='I have some leftover chicken,rice, an onion, and some tortillas. What can I make?', additional_kwargs={}, response_metadata={}, id='7e20d233-2b8f-45ae-a28f-294cc40ba350'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user has chicken, rice, an onion, and tortillas. Let\'s see. They want to know what they can make with these ingredients. I need to find recipes that use all or most of these items. Let me start by thinking of possible dishes. \n\nChicken and rice is a common combo. Maybe a stir-fry? But they also have tortillas. Tortillas make me think of wraps or burritos. Oh, maybe chicken tortilla soup? Wait, but they have rice. Alternatively, a chicken and rice casserole with tortillas on the side? Or perhaps a chicken fajita with rice. Wait, fajitas usually use peppers too, but they have an onion. Maybe chicken and onion rice bowls with tortillas as wraps.\n\nAlternatively, a chicken quesadilla with rice an

In [16]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [17]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [18]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What can i make with these ingredients?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

config = {"configurable": {"thread_id": "2"}}
response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

Based on the ingredients in your pantry (pasta, chicken breast, canned tomatoes, Parmesan cheese, onion, garlic, and herbs), here's a simple recipe idea:

### **Cheesy Tomato Chicken Pasta**  
**Instructions**:  
1. Cook your pasta (penne, spaghetti, or fusilli) according to package instructions.  
2. Sauté chopped onion and garlic in olive oil until softened.  
3. Add crushed tomatoes (San Marzano), dried herbs (basil, oregano, thyme), and a pinch of red pepper flakes. Simmer for 10 minutes.  
4. Add cooked chicken breast (seasoned with salt and pepper) to the sauce and simmer for 5-7 minutes.  
5. Toss the sauce with the cooked pasta and grated Parmesan cheese.  
6. Optional: Top with crispy panko (from the box) and extra Parmesan for a crunchy finish.  

This dish combines the flavors of tomato sauce, herbs, and cheesy richness. Let me know if you'd like more details! 🍝
